# 05 Test-set audit and final refit

Workflow steps 6-8. **Run this notebook once.** It loads the test set for the first and only time, scores the single model selected in notebook 03 at the operating point fixed in notebook 04, and then refits that exact specification on all the data. If the audit fails, report the failure. Do not change the model and re-score: the test set would then become another validation set.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import (average_precision_score, roc_auc_score, roc_curve,
                             precision_recall_curve, confusion_matrix, accuracy_score)

from wine_utils import (load_train, transform, build_selected, expected_cost,
                        FEATURES, C_FN_BASE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

spec = json.load(open('results/selected_model.json'))
op = json.load(open('results/operating_point.json'))
X_train, y_train = load_train()
spec, op

## Audit pass criteria (fixed before the test set is loaded)

The audit passes only if **both** hold:
1. **Test agrees with CV:** test PR-AUC lies within the CV mean ± 2 SE from notebook 03.
2. **Beats the baselines:** test PR-AUC is above the share of good wines in the training set, test ROC-AUC is above 0.5, and test accuracy at threshold 0.5 is above the majority-class accuracy (predicting *not good* for every wine).

Operating-point results (sensitivity, specificity and cost at the threshold from notebook 04) are reported but are not pass/fail criteria.

In [ ]:
lo = spec['cv_pr_auc_mean'] - 2 * spec['cv_pr_auc_se']
hi = spec['cv_pr_auc_mean'] + 2 * spec['cv_pr_auc_se']
pr_baseline = y_train.mean()
print(f'Criterion 1: test PR-AUC in [{lo:.3f}, {hi:.3f}]')
print(f'Criterion 2: PR-AUC > {pr_baseline:.3f}, ROC-AUC > 0.5, accuracy > majority-class accuracy')

## Step 6: fit the selected specification on the full training set

In [ ]:
model = build_selected(spec).fit(X_train, y_train)
model

## Step 7: score the test set, once

In [ ]:
test = pd.read_csv('datasets/winequality-white-test.csv')
X_test, y_test = transform(test[FEATURES]), (test['quality'] >= 7).astype(int)
prob = model.predict_proba(X_test)[:, 1]

majority = int(y_train.mean() >= 0.5)
result = {
    'PR-AUC': average_precision_score(y_test, prob),
    'ROC-AUC': roc_auc_score(y_test, prob),
    'Accuracy @0.5': accuracy_score(y_test, prob >= 0.5),
    'Majority-class accuracy': accuracy_score(y_test, np.full(len(y_test), majority)),
}
passed_cv = lo <= result['PR-AUC'] <= hi
passed_base = (result['PR-AUC'] > pr_baseline and result['ROC-AUC'] > 0.5
               and result['Accuracy @0.5'] > result['Majority-class accuracy'])
print('Criterion 1 (agrees with CV):', 'PASS' if passed_cv else 'FAIL')
print('Criterion 2 (beats baselines):', 'PASS' if passed_base else 'FAIL')
print('AUDIT:', 'PASSED' if passed_cv and passed_base else 'FAILED')
pd.Series(result).round(3)

### CV estimate against the held-out test result

In [ ]:
pd.DataFrame({
    'CV mean': [spec['cv_pr_auc_mean'], spec['cv_roc_auc_mean']],
    'CV SE': [spec['cv_pr_auc_se'], spec['cv_roc_auc_se']],
    'Test': [result['PR-AUC'], result['ROC-AUC']],
}, index=['PR-AUC', 'ROC-AUC']).assign(**{'Test - CV (in SEs)': lambda d: (d['Test'] - d['CV mean']) / d['CV SE']}).round(3)

### Operating point on the test set
Threshold fixed in notebook 04 (base case $c_{FN}/c_{FP} = 3$).

In [ ]:
t = op['threshold']
cm = confusion_matrix(y_test, prob >= t)
tn, fp, fn, tp = cm.ravel()
pi = y_test.mean()

fig, ax = plt.subplots(figsize=(4, 3.5))
sns.heatmap(cm, annot=True, fmt='d', cbar=False, cmap='Blues', ax=ax,
            xticklabels=['not good', 'good'], yticklabels=['not good', 'good'])
ax.set(xlabel='Predicted', ylabel='Actual', title=f'Test set, threshold {t:g}')
plt.tight_layout()
plt.show()

pd.DataFrame({
    'CV (training)': [op['cv_sensitivity'], op['cv_specificity'], op['cv_cost_per_wine']],
    'Test': [tp / (tp + fn), tn / (tn + fp), expected_cost(y_test, prob, t, C_FN_BASE)],
}, index=['Sensitivity', 'Specificity', 'Cost per wine']).round(3)

In [ ]:
pd.Series({
    f'Model @ {t:g}': expected_cost(y_test, prob, t, C_FN_BASE),
    'Model @ 0.5': expected_cost(y_test, prob, 0.5, C_FN_BASE),
    'Send every batch to the panel': 1 - pi,
    'Send none': C_FN_BASE * pi,
}, name=f'Test cost per wine (c_FN/c_FP = {C_FN_BASE})').round(3)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, prob)
prec, rec, _ = precision_recall_curve(y_test, prob)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, lw=2, label=f"Test (AUC {result['ROC-AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], color='grey', ls='--', lw=0.8, label='No skill')
axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title='Test ROC curve')
axes[1].plot(rec, prec, lw=2, label=f"Test (AP {result['PR-AUC']:.3f})")
axes[1].axhline(pi, color='grey', ls='--', lw=0.8, label=f'No skill ({pi:.2f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Test precision-recall curve')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 8: refit the same specification on all data (train + test)

Only the coefficients are re-estimated. The predictors, transformations, model family and threshold stay frozen. sklearn gives no standard errors, so the same unpenalised logistic model is refitted in `statsmodels`. Predictors are standardised over the full data, so each odds ratio is per one-SD increase. The performance figures to report remain the step 7 test results.

In [ ]:
assert spec['name'] == 'Logistic (all 11)', 'Step 8 below is written for the unpenalised logistic specification'

X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
Z_all = (X_all - X_all.mean()) / X_all.std()

final = sm.Logit(y_all, sm.add_constant(Z_all)).fit(disp=0)
print(final.summary())

In [ ]:
ci = final.conf_int()
pd.DataFrame({
    'coef (log-odds per SD)': final.params,
    'odds ratio': np.exp(final.params),
    'OR 95% CI low': np.exp(ci[0]),
    'OR 95% CI high': np.exp(ci[1]),
    'p-value': final.pvalues,
}).drop('const').sort_values('odds ratio', ascending=False).round(3)